# 08 · Which genes explain the lipid changes

Following the tutorial notebook [`08_multimodal_xgb_shap_go_student.ipynb`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/blob/main/notebooks/level3/08_multimodal_xgb_shap_go_student.ipynb).

**Kernel:** `cajal-lipidomics`

**What I'm doing here**
- Per-pixel: borrow MERFISH genes + cell type with a KD-tree ball query (Mog vs HexCer 42:2)
- Per-region: lipid-change matrix joined to region-averaged MERFISH
- 20 NMF gene programs -> one XGBoost per ion, held-out Pearson r, SHAP
- Gene ontology on the top programs, permutation null, assemble the figure

**My notes / what surprised me**
- ...

In [ ]:
# ---- setup: the same first cell in every notebook ---------------------------------------

# My own little package. `paths` knows where the tutorial clone, its provided data,
# and my own outputs live, so I never type "../../data/..." by hand.
from lipid_project import paths

paths.ensure_dirs()   # creates data/derived/, figures/, results/ if they're missing (safe to re-run)
paths.summary()       # prints where everything points + which stages I've built so far.
                      # If a line starts with "--", that thing is missing: fix it before going on.

# The usual scientific stack
import numpy as np                 # arrays and math
import pandas as pd                # tables (obs / var are pandas DataFrames)
import matplotlib.pyplot as plt    # plotting
import anndata as ad               # the AnnData container: X (pixels x lipids) + obs + var + obsm + layers

# The tutorial's helper package (installed from my clone in external/).
# Rule from the course: before calling any cl.<module>.<function>, open
# external/lipidomics_tutorial_cajalcourse/src/cajal_lipidomics/<module>.py and read it.
import cajal_lipidomics as cl
cl.style.set_style()               # the course's figure defaults, so plots look clean and consistent

# If this prints without a red error, the kernel is right. A ModuleNotFoundError here almost
# always means the wrong kernel: re-pick "cajal-lipidomics" in the top-right kernel picker.
print("ready. cajal_lipidomics", cl.__version__)

## load

In [ ]:
# ---- load the previous stage --------------------------------------------------------------
# My NB06 output: pixels with uMAIA lipids, Allen region, and my lipizones.
adata = ad.read_h5ad(paths.stage("clustered"))

# The second modality, Allen MERFISH, from the tutorial's data bundle. Different mice,
# different lab, but registered into the same Allen CCF, which is the whole bridge:
#   - per-cell MERFISH for this plane: CCF coords + cell type + 500 measured genes per cell
#     (for the per-pixel KD-tree "borrow genes from nearby cells" part)
merfish_cells_path  = paths.tutorial_data("merfish_plane.parquet")
#   - region-averaged, imputed MERFISH: one row per Allen region, 8460 gene columns
#     (for the per-region XGBoost / SHAP part, joined on the region acronym)
merfish_region_path = paths.tutorial_data("avemerfish_imputed_named.parquet")

# Gene ontology files (go-basic.obo + mouse gene2go), pre-downloaded in the data bundle
GO_DIR = paths.GO

print("pixels x lipids:", adata.shape)
print("MERFISH files found:", merfish_cells_path.exists(), merfish_region_path.exists())
print("GO folder found:", GO_DIR.exists())

## my work

In [ ]:
# my code from here on, one small step per cell, and look at the output each time